# Moment Experiment M1

Source-video-only comparison of frozen BTC technical-keyframe anchors against query-local raw-video CLIP refinement. This is an internal AI pseudo-GT experiment, not an official metric or production Semantic Moment Localizer.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
STAGE1_INPUT = os.environ.get("AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle")
DATASET_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
STAGE1B_INPUT = os.environ.get("AIC_STAGE1B_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports")
STAGE1E_INPUT = os.environ.get("AIC_STAGE1E_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze")
CLIP_INPUT = os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
OPUS_INPUT = os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
BENCHMARK_INPUT = os.environ.get("AIC_M1_BENCHMARK_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle")
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_moment_m1")
ZIP_PATH = Path("/kaggle/working/triage_eg_moment_m1_bundle.zip")
print({"dataset": str(DATASET_ROOT), "benchmark_input": BENCHMARK_INPUT, "output": str(OUTPUT_ROOT)})

In [ ]:
def find_marker_roots(root, marker, max_depth=4):
    root = Path(root)
    matches, frontier = [], [(root, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if (current / marker).is_file():
            matches.append(current)
            continue
        if depth < max_depth and current.is_dir():
            frontier.extend((child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir())
    return matches

def resolve_root(root, marker, max_depth=4):
    matches = find_marker_roots(root, marker, max_depth)
    if len(matches) != 1:
        raise RuntimeError(f"Expected one root containing {marker} below {root}; found {matches}")
    return matches[0]

def resolve_input_file(root, filename, search_root=Path("/kaggle/input")):
    requested = Path(root)
    if requested.is_file() and requested.name == filename:
        return requested
    matches = find_marker_roots(requested, filename, 6)
    if not matches:
        matches = find_marker_roots(search_root, filename, 6)
    paths = sorted({match / filename for match in matches})
    if len(paths) != 1:
        raise RuntimeError(f"Expected one {filename}; found {paths}")
    return paths[0]

STAGE1_ROOT = resolve_root(STAGE1_INPUT, "stage1_summary.json")
STAGE1B_ROOT = resolve_root(STAGE1B_INPUT, "stage1b_summary.json")
STAGE1E_ROOT = resolve_root(STAGE1E_INPUT, "language_path_contract.json")
CLIP_ROOT = resolve_root(CLIP_INPUT, "manifests/asset_manifest.json")
OPUS_ROOT = resolve_root(OPUS_INPUT, "manifests/asset_manifest.json")
BENCHMARK_PATH = resolve_input_file(BENCHMARK_INPUT, "rt2_ai_benchmark.jsonl")
if not DATASET_ROOT.is_dir():
    raise RuntimeError(f"Missing dataset root: {DATASET_ROOT}")
print({"stage1": str(STAGE1_ROOT), "stage1b": str(STAGE1B_ROOT), "stage1e": str(STAGE1E_ROOT), "clip": str(CLIP_ROOT), "opus": str(OPUS_ROOT), "benchmark": str(BENCHMARK_PATH)})

In [ ]:
from triage_eg.experiments.moment_m1 import M1RunnerConfig, load_m1_settings, preflight_moment_m1
from triage_eg.experiments.reference_rt2 import load_rt2_benchmark
from triage_eg.retrieval.stage2 import config_from_yaml

SETTINGS = load_m1_settings(REPO_DIR / "configs/experiments/moment_m1.yaml")
QUERIES = load_rt2_benchmark(BENCHMARK_PATH)
STAGE2 = config_from_yaml(REPO_DIR / "configs/retrieval/stage2_operational_runtime.yaml", stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, stage1e_root=STAGE1E_ROOT, clip_asset_root=CLIP_ROOT, translator_asset_root=OPUS_ROOT, output_root=OUTPUT_ROOT / "_stage2_control", stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml", build_git_commit=COMMIT)
CONFIG = M1RunnerConfig(STAGE2, DATASET_ROOT, BENCHMARK_PATH, OUTPUT_ROOT, SETTINGS)
PREFLIGHT = preflight_moment_m1(CONFIG, QUERIES)
print(json.dumps(PREFLIGHT, indent=2))

In [ ]:
from triage_eg.experiments.moment_m1 import run_moment_m1

RESULT = run_moment_m1(CONFIG, QUERIES)
print(json.dumps(RESULT, indent=2))

In [ ]:
METRICS = json.loads((OUTPUT_ROOT / "m1_metrics.json").read_text(encoding="utf-8"))
for scope in ("ALL_EVENTS", "REFERENCE_REACHABLE_EVENTS"):
    values = METRICS[scope]
    print(scope, json.dumps(values, indent=2))

In [ ]:
from IPython.display import Image, display

for path in sorted((OUTPUT_ROOT / "visuals").glob("*_ab.jpg"))[:6]:
    print(path.stem)
    display(Image(filename=str(path)))

In [ ]:
from triage_eg.experiments.moment_m1 import create_m1_bundle

create_m1_bundle(OUTPUT_ROOT, ZIP_PATH)
print("DOWNLOAD ZIP:", ZIP_PATH, "size_bytes=", ZIP_PATH.stat().st_size)
print("M1_IMPLEMENTATION_STATUS = COMPLETE")
print("M1_REAL_EXPERIMENT_STATUS = COMPLETE")
print("M1_QUALITY_DECISION = NOT_EVALUATED")